# 
Importing our Chat Models:


get API from local env

In [1]:
import os
import langchain
# to import environment variables from .env file
from dotenv import load_dotenv

load_dotenv()

api_key = os.environ.get("GEMINI_API_KEY")

if not api_key:
    raise ValueError("GEMINI_API_KEY is not set in the environment variables.")

#
Testing the model

In [2]:

from langchain_google_genai import ChatGoogleGenerativeAI
model = ChatGoogleGenerativeAI(model="gemini-3.6-flash", google_api_key=api_key)

# 
making prompt templates




1-

#
2-

#
3- Example applied on RAG with Chat Models

check langchain that its installed

In [ ]:
%pip install langchain-text-splitters
%pip install langchain-community

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.1.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


# Indexing Pipeline:

1- Load Documents or Text

In [2]:
from langchain_text_splitters import MarkdownHeaderTextSplitter
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

paths = [
    r"C:\Users\oasm2\OneDrive\Documents\2nd project\Just_Practicing\omar_env\data_cleaning_task1_part1.md",
    r"C:\Users\oasm2\OneDrive\Documents\2nd project\Just_Practicing\omar_env\eda_task1_part2.md",
    r"C:\Users\oasm2\OneDrive\Documents\2nd project\Just_Practicing\omar_env\supervised_ml_model_task3.md",
    r"C:\Users\oasm2\OneDrive\Documents\2nd project\Just_Practicing\omar_env\unsupervised_ml_model_task4.md",
]

documents = []
for p in paths:
    loader = TextLoader(p, encoding="utf-8")
    documents.extend(loader.load())

print(f"Loaded : {len(documents)} documents")
for doc in documents:
    print(" -", doc.metadata["source"])


# STEP 2a: Split by markdown headers first

headers_to_split_on = [
    ("#", "H1"),
    ("##", "H2"),
    ("###", "H3"),
]

md_splitter = MarkdownHeaderTextSplitter(headers_to_split_on=headers_to_split_on)

header_chunks = []
for doc in documents:
    chunks = md_splitter.split_text(doc.page_content)
    for chunk in chunks:
        chunk.metadata["source"] = doc.metadata["source"]  # keep track of which file this came from
    header_chunks.extend(chunks)

print(f"\nAfter header split: {len(header_chunks)} chunks")


# STEP 2b: Further split any oversized sections by character count

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=100,
    separators=["\n\n", "\n", " ", ""]
)

final_chunks = text_splitter.split_documents(header_chunks)

print(f"After size-limit split: {len(final_chunks)} final chunks")


# STEP 3: Sanity checks before moving to embedding

from collections import Counter

sources = Counter(chunk.metadata["source"] for chunk in final_chunks)
print("\nChunks per file:")
for source, count in sources.items():
    print(f"  {source}: {count} chunks")

# Preview a sample chunk to confirm content + metadata look right
print("\n--- Sample chunk ---")
print("Metadata:", final_chunks[0].metadata)
print("Content preview:", final_chunks[0].page_content[:300])

c:\Users\oasm2\OneDrive\Documents\2nd project\Just_Practicing\omar_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loaded : 4 documents
 - C:\Users\oasm2\OneDrive\Documents\2nd project\Just_Practicing\omar_env\data_cleaning_task1_part1.md
 - C:\Users\oasm2\OneDrive\Documents\2nd project\Just_Practicing\omar_env\eda_task1_part2.md
 - C:\Users\oasm2\OneDrive\Documents\2nd project\Just_Practicing\omar_env\supervised_ml_model_task3.md
 - C:\Users\oasm2\OneDrive\Documents\2nd project\Just_Practicing\omar_env\unsupervised_ml_model_task4.md

After header split: 64 chunks
After size-limit split: 108 final chunks

Chunks per file:
  C:\Users\oasm2\OneDrive\Documents\2nd project\Just_Practicing\omar_env\data_cleaning_task1_part1.md: 17 chunks
  C:\Users\oasm2\OneDrive\Documents\2nd project\Just_Practicing\omar_env\eda_task1_part2.md: 20 chunks
  C:\Users\oasm2\OneDrive\Documents\2nd project\Just_Practicing\omar_env\supervised_ml_model_task3.md: 39 chunks
  C:\Users\oasm2\OneDrive\Documents\2nd project\Just_Practicing\omar_env\unsupervised_ml_model_task4.md: 32 chunks

--- Sample chunk ---
Metadata: {'H1': 'D

# EMBEDDING PART :
## Getting our chunks to be vectors through Embedding model sentence tranformer

In [13]:
%pip install langchain-huggingface 
%pip install -q -U sentence-transformers

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.1.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


^C
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.1.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [5]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
# just to check the embedding dimensions, we can embed a single document and print the length of the resulting vector
# docs_embedded = embeddings.embed_documents([chunk.page_content for chunk in final_chunks])

# print(f"Number of dimensions in a embedded doc: {len(docs_embedded[0])}")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1808.54it/s]


## Moving our chunks that embedded in vectors to be in Vector Database through (Chroma)

In [19]:
%pip install langchain-chromadb

Note: you may need to restart the kernel to use updated packages.


ERROR: Could not find a version that satisfies the requirement langchain-chromadb (from versions: none)

[notice] A new release of pip is available: 25.1.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip
ERROR: No matching distribution found for langchain-chromadb


In [8]:
from langchain_community.vectorstores import Chroma
import shutil
shutil.rmtree("./chroma_db", ignore_errors=True)

vector_store = Chroma.from_documents(
    documents=final_chunks,
    embedding=embeddings,
    persist_directory="./chroma_db"
)

len(vector_store.get()["ids"])


108

In [4]:
# to not embedding the documents again, we can save the vector store to disk and load it later
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

vectorstore = Chroma(
    persist_directory="./chroma_db",
    embedding_function=embeddings
)

print(f"Loaded {vectorstore._collection.count()} chunks from disk")

c:\Users\oasm2\OneDrive\Documents\2nd project\Just_Practicing\omar_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4822.55it/s]


Loaded 108 chunks from disk


## TEST RETRIEVAL

In [6]:
# Sanity check: perform a similarity search to see if the vector store is working as expected (TEST RETRIEVAL)
query = "Why did DBSCAN perform worse than K-Means?"
results = vectorstore.similarity_search(query, k=3)

for i, doc in enumerate(results):
    print(f"\n--- Result {i+1} ---")
    print("Source:", doc.metadata.get("source"))
    print("Section:", doc.metadata.get("H2") or doc.metadata.get("H3"))
    print("Content:", doc.page_content[:300])


--- Result 1 ---
Source: C:\Users\oasm2\OneDrive\Documents\2nd project\Just_Practicing\omar_env\unsupervised_ml_model_task4.md
Section: Final comparison: K-Means vs. Agglomerative vs. DBSCAN — which is the best model
Content: 2. **Why DBSCAN underperformed here specifically:** DBSCAN is designed to find naturally dense regions separated by sparser gaps, and works best when a dataset has that kind of density structure. This dataset's four clustering features don't show that kind of gap-separated density — consistent with 

--- Result 2 ---
Source: C:\Users\oasm2\OneDrive\Documents\2nd project\Just_Practicing\omar_env\unsupervised_ml_model_task4.md
Section: DBSCAN — third clustering algorithm, run and evaluated
Content: DBSCAN was run using `eps=0.35` (based on the K-distance graph estimate above) and `min_samples=5` (matching the 5-neighbor count used to build that graph), on the same scaled four-feature set (CustomerAge, ProductPrice, PurchaseFrequency, CustomerSatisfaction) used for 

## Getting our vectorstore variable to be our retreiver that retreives relevant documents for the query question  

In [6]:
def build_prompt(query, retrieved_docs):     # <-- This function constructs a prompt for the model using the query and retrieved documents
    context = "\n\n".join([doc.page_content for doc in retrieved_docs])
    
    prompt = f"""You are a helpful assistant that answers questions using the provided context in my own project 
            project context. If the answer is not in the context, say: I don't know.

Context:
{context}

Question: {query}
Answer:"""
    
    return prompt

def get_text(response):         # <-- This function extracts the text content from the model's response, handling different response formats
    if isinstance(response.content, str):
        return response.content
    elif isinstance(response.content, list):
        return "".join(block["text"] for block in response.content if block.get("type") == "text")
    return str(response.content)


def answer_question(query, k=3):    # <-- this function retrieves relevant documents based on the query, constructs a prompt, and generates an answer using the model
    retrieved_docs = vectorstore.similarity_search(query, k=k)
    prompt = build_prompt(query, retrieved_docs)
    response = model.invoke(prompt)   # <-- Generating the answer using the model with the constructed prompt
    return get_text(response), retrieved_docs     
    
    
           
    

# Testing the answer_question function with a sample query

answer, sources = answer_question("Reinforcment Learning results")

print("ANSWER:")
print(answer)

print("\nSOURCES USED:")
for doc in sources:
    print(" -", doc.metadata.get("source"), "|", doc.metadata.get("H2") or doc.metadata.get("H3"))

ANSWER:
I don't know.

SOURCES USED:
 - C:\Users\oasm2\OneDrive\Documents\2nd project\Just_Practicing\omar_env\supervised_ml_model_task3.md | Models built and their individual results
 - C:\Users\oasm2\OneDrive\Documents\2nd project\Just_Practicing\omar_env\supervised_ml_model_task3.md | Models built and their individual results
 - C:\Users\oasm2\OneDrive\Documents\2nd project\Just_Practicing\omar_env\supervised_ml_model_task3.md | Purpose of this notebook
